In [1]:
!pip install dagshub mlflow statsforecast -q

import pandas as pd
import numpy as np
import dagshub
import mlflow
from statsforecast import StatsForecast
from statsforecast.models import AutoARIMA
import warnings
warnings.filterwarnings('ignore')

REPO_OWNER = "ejoba22"  
REPO_NAME = "walmart-sales-forecasting"

dagshub.init(repo_owner=REPO_OWNER, repo_name=REPO_NAME, mlflow=True)

mlflow.set_tracking_uri(f"https://dagshub.com/{REPO_OWNER}/{REPO_NAME}.mlflow")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.7/49.7 kB 1.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.5/50.5 kB 3.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 2.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 273.1/273.1 kB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 90.9 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 87.0 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 59.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 354.6/354.6 kB 18.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 348.6/348.6 kB 24.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.2/68.2 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 281.0/281.0 kB 18.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━

❗❗❗ AUTHORIZATION REQUIRED ❗❗❗

Output()



Open the following link in your browser to authorize the client:
https://dagshub.com/login/oauth/authorize?state=9240143f-6e4c-4216-bf98-b21ac4c25df8&client_id=32b60ba385aa7cecf24046d8195a71c07dd345d9657977863b52e7748e0f0f28&middleman_request_id=a7c234c01b5ef715798d8d250e3785e75002f33664bbf03716bde5995b1fc6da




Accessing as tvada22

Initialized MLflow to track repo "ejoba22/walmart-sales-forecasting"

Repository ejoba22/walmart-sales-forecasting initialized!

In [5]:
# 2. DATA LOADING & FORMATTING
DATA_BASE_PATH = '/kaggle/input/notebooks/elenejobava/walmart-eda-feature-engineering/'
train_fe = pd.read_parquet(DATA_BASE_PATH + 'train_features.parquet')

# Clean and format for Nixtla (Same as Deep Learning)
if 'Type' in train_fe.columns:
    train_fe = train_fe.drop(columns=['Type'])

df_stats = train_fe.copy()
df_stats = df_stats.rename(columns={'Store_Dept': 'unique_id', 'Date': 'ds', 'Weekly_Sales': 'y'})
df_stats['weight'] = np.where(df_stats['IsHoliday'] == 1, 5, 1)

df_stats = df_stats.fillna(0)
df_stats = df_stats.sort_values(['unique_id', 'ds']).reset_index(drop=True)

# Time-Series Split (12 weeks)
split_date = df_stats['ds'].max() - pd.Timedelta(weeks=12)
train_df = df_stats[df_stats['ds'] <= split_date]
val_df = df_stats[df_stats['ds'] > split_date]

# Filter out short time series (AutoARIMA needs history to find seasonality)
series_lengths = train_df['unique_id'].value_counts()
valid_ids = series_lengths[series_lengths >= 52].index
train_df = train_df[train_df['unique_id'].isin(valid_ids)]
val_df = val_df[val_df['unique_id'].isin(valid_ids)]


In [3]:
# 3. METRICS
def calculate_wape(y_true, y_pred):
    return (np.sum(np.abs(y_true - y_pred)) / np.sum(np.abs(y_true))) * 100

def calculate_wmae(y_true, y_pred, weights):
    return np.sum(weights * np.abs(y_true - y_pred)) / np.sum(weights)

In [4]:
# 4. ARCHITECTURE SETUP & TRAINING

# Define the AutoARIMA model
models = [
    AutoARIMA(season_length=52) # 52 weeks in a year forces the S in SARIMA
]

# Initialize StatsForecast
sf = StatsForecast(
    models=models,
    freq='W-FRI',   # Friday alignment
    n_jobs=-1       # Use all Kaggle CPU cores for speed
)


# Pass strictly the 3 required columns so it doesn't ask for future feature values
train_df_arima = train_df[['unique_id', 'ds', 'y']]

# fit_predict automatically trains the math and generates the 12-week horizon
val_preds_df = sf.fit_predict(df=train_df_arima, h=12).reset_index()

# Merge predictions back with the original val_df (which still has our 'weight' column)
results = val_df[['unique_id', 'ds', 'y', 'weight']].merge(val_preds_df, on=['unique_id', 'ds'], how='inner')

In [6]:
# 5. MLFLOW LOGGING
mlflow.set_experiment("AutoARIMA_Training")

with mlflow.start_run(run_name="AutoARIMA_Baseline"):
    wmae_arima = calculate_wmae(results['y'], results['AutoARIMA'], results['weight'])
    wape_arima = calculate_wape(results['y'], results['AutoARIMA'])
    
    mlflow.log_param("architecture", "SARIMA")
    mlflow.log_param("season_length", 52)
    mlflow.log_metric("validation_WMAE", wmae_arima)
    mlflow.log_metric("validation_WAPE_percent", wape_arima)
    
    print(f"Validation WMAE: {wmae_arima:.2f}")
    print(f"Human-Readable Error (WAPE): {wape_arima:.2f}%")

Validation WMAE: 1513.97
Human-Readable Error (WAPE): 9.54%
🏃 View run AutoARIMA_Baseline at: https://dagshub.com/ejoba22/walmart-sales-forecasting.mlflow/#/experiments/13/runs/6813d113de8f45a99a7a62b5d720d172
🧪 View experiment at: https://dagshub.com/ejoba22/walmart-sales-forecasting.mlflow/#/experiments/13


In [2]:
## 6. AUTOARIMA PIPELINE & REGISTRY
from sklearn.base import BaseEstimator, RegressorMixin
from statsforecast import StatsForecast
from statsforecast.models import AutoARIMA
import mlflow.sklearn
import pandas as pd

# 1. Create a Custom Wrapper for StatsForecast
class ClassicalPipeline(BaseEstimator, RegressorMixin):
    def __init__(self, season_length=52):
        self.season_length = season_length
        self.sf_ = None
        self.valid_ids_ = None

    def fit(self, X, y):
        df = X.copy()
        df['y'] = y
        # Create the unique ID Nixtla requires
        df['unique_id'] = df['Store'].astype(str) + '_' + df['Dept'].astype(str)
        df = df.rename(columns={'Date': 'ds'})

        # Drop departments that don't have enough history to calculate 52-week seasonality
        series_lengths = df['unique_id'].value_counts()
        self.valid_ids_ = series_lengths[series_lengths >= self.season_length].index
        df = df[df['unique_id'].isin(self.valid_ids_)].reset_index(drop=True)

        # Keep only the required columns so it doesn't crash expecting future exogenous features
        df = df[['unique_id', 'ds', 'y']]

        models = [AutoARIMA(season_length=self.season_length)]
        self.sf_ = StatsForecast(models=models, freq='W-FRI', n_jobs=-1)
        self.sf_.fit(df=df)
        
        return self

    def predict(self, X):
        preds = self.sf_.predict(h=12).reset_index()

        # Format the incoming raw Kaggle test set
        df_test = X.copy()
        df_test['unique_id'] = df_test['Store'].astype(str) + '_' + df_test['Dept'].astype(str)
        df_test = df_test.rename(columns={'Date': 'ds'})

        # Merge the predictions onto the exact rows Kaggle is asking for
        merged = df_test.merge(preds, on=['unique_id', 'ds'], how='left')

        # Fill predictions for any dropped short-series departments with 0
        return merged['AutoARIMA'].fillna(0).values

# 2. Initialize the pipeline
arima_pipeline = ClassicalPipeline(season_length=52)


# 7. TRAIN & REGISTER IN MLFLOW
# Load raw, un-engineered data 
raw_train = pd.read_csv('/kaggle/input/competitions/walmart-recruiting-store-sales-forecasting/train.csv.zip', parse_dates=['Date'])

# We only need the base columns to prove the pipeline works
raw_X = raw_train[['Store', 'Dept', 'Date']]
raw_y = raw_train['Weekly_Sales']

mlflow.set_experiment("AutoARIMA_Training")
with mlflow.start_run(run_name="Final_AutoARIMA_Pipeline"):
    
    # Explicitly log parameters for the dashboard
    mlflow.log_param("architecture", "AutoARIMA")
    mlflow.log_param("season_length", 52)
    
    # Train the pipeline directly on the raw data
    arima_pipeline.fit(raw_X, raw_y)
    
    # Log the pipeline to MLflow using CloudPickle
    mlflow.sklearn.log_model(
        sk_model=arima_pipeline,
        artifact_path="model",
        serialization_format="cloudpickle" 
    )


2026/07/12 15:26:17 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/07/12 15:26:18 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run Final_AutoARIMA_Pipeline at: https://dagshub.com/ejoba22/walmart-sales-forecasting.mlflow/#/experiments/13/runs/f04691eb10634621947ec08d438454d3
🧪 View experiment at: https://dagshub.com/ejoba22/walmart-sales-forecasting.mlflow/#/experiments/13
